# LLM Structured Output with Outlines

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/Building_with_Deep_Learning/01-llms/05_llm_structured_output_exercise.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

Free-form generation is messy to parse. **Outlines** constrains the LLM so every response matches a type you choose — an `int`, a `Literal` label, or a Pydantic model.


**Goal:** Extract guaranteed-valid structured data from product reviews, then apply the same pattern to a resume PDF.

**Topics:** simple types (`int`), `Literal` classification, Pydantic schemas, PDF → markdown with PDF4LLM.


In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/Building_with_Deep_Learning/01-llms"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Imports


In [ ]:
%pip install -qqq outlines transformers pymupdf4llm


In [ ]:
import os
import sys
import logging
from enum import Enum
from pathlib import Path
from typing import Literal

from huggingface_hub.utils import disable_progress_bars
from IPython.display import Markdown, display
from transformers.utils import logging as transformers_logging
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import outlines

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
disable_progress_bars()
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)


# Load the model

Wrap a small instruction-tuned model with Outlines. The same `model(prompt, output_type)` call works for simple types, `Literal` labels, and Pydantic models.

While it downloads, skim the [Qwen2.5-0.5B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).


In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

hf_model = AutoModelForCausalLM.from_pretrained(model_name)
hf_tokenizer = AutoTokenizer.from_pretrained(model_name)

model = outlines.from_transformers(hf_model, hf_tokenizer)


# NLP Scenario
You are an analyst for a marketing company that just launched a new product suite of mobile devices. You have data from product reviews of the new product.

In the generation lab you drafted free-form marketing copy. The client now wants a **structured report**: sentiment labels, star ratings, and lists of pros and cons they can drop into a spreadsheet — not a paragraph they have to parse by hand.

##### Product Reviews
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6. "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."


In [ ]:
# data setup
reviews = [
    "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!",
    "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.",
    "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.",
    "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.",
    "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.",
    "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities.",
]

long_review = """I've been using the TechWave X1 for three months. The battery easily lasts a full day and the camera is excellent in daylight. On the downside, the phone runs hot during gaming and the speakers are tinny. Overall it's a solid device if you can live with the heat."""


# Simple structured outputs

Pass a Python type as the second argument. Outlines constrains generation so the result is that type — no regex, no `json.loads`.

```python
model(prompt, int, max_new_tokens=5)
```


In [ ]:
# extract an implied star rating (1-5) as an integer
stars = model(
    f"On a scale of 1 to 5, how many stars does this review imply? Review: {reviews[0]}",
    int,
    max_new_tokens=5,
)
print(stars)


# Multiple-choice with Literal

`Literal[...]` restricts the model to one of a fixed set of labels. That is classification without a separate classifier pipeline.

```python
from typing import Literal

model(prompt, Literal["Positive", "Negative", "Neutral"])
```


In [ ]:
# classify one review
sentiment = model(
    f"Analyze the sentiment of this review: {reviews[0]}",
    Literal["Positive", "Negative", "Neutral"],
    max_new_tokens=10,
)
print(sentiment)


# Try it!
1. Classify **all six** reviews with `Literal["Positive", "Negative", "Neutral"]`.
2. Compare the labels to your own reading of the reviews. How did the model do on the average review (item 4)?


In [ ]:
# INSERT YOUR CODE HERE


# Complex structures with Pydantic

For nested records, define a Pydantic `BaseModel` (and an `Enum` if you want a closed set of ratings). Pass the class as `output_type`, then parse the JSON string back into a Python object:

```python
from pydantic import BaseModel
from enum import Enum

class Rating(Enum):
    poor = 1
    fair = 2
    good = 3
    excellent = 4

class ProductReview(BaseModel):
    rating: Rating
    pros: list[str]
    cons: list[str]
    summary: str

raw = model(prompt, ProductReview, max_new_tokens=200)
review = ProductReview.model_validate_json(raw)
```

The lecture used this pattern on an XPS 13 review. Apply it to the TechWave `long_review`.


# Try it!
1. Define a `Rating` enum and a `ProductReview` model with `rating`, `pros`, `cons`, and `summary`.
2. Call `model` on `long_review` with `ProductReview` as the output type (`max_new_tokens=200`).
3. Validate with `ProductReview.model_validate_json` and print the rating name, pros, cons, and summary.


In [ ]:
# INSERT YOUR CODE HERE


# Your CV as structured data

Product reviews were a warm-up. A resume is the same problem at document scale: unstructured PDF in, fields you can store or query out.

The pipeline is:

1. Parse the PDF to markdown with PDF4LLM
2. Define a Pydantic `Resume` schema
3. Constrain generation with Outlines so the model fills that schema


### Parse PDF into Markdown with PDF4LLM

Real extraction pipelines often start with PDFs, not ready-made text. [pymupdf4llm](https://docs.pdf4llm.com/python/api/to_markdown) converts a PDF to markdown so the LLM can read it.

Extract clean, structured content from any document — ready for LLMs, RAG pipelines, and AI applications.

[PDF4LLM](https://docs.pdf4llm.com/) gives you something you can actually use — [clean Markdown](https://docs.pdf4llm.com/python/guides/extract-Markdown), [structured JSON](https://docs.pdf4llm.com/python/guides/extract-JSON), or [plain text](https://docs.pdf4llm.com/python/guides/extract-Text), with:

1. reading order preserved
2. tables intact
3. and images handled

..in a single function call.


In [ ]:
source = "my_resume.pdf"

In [ ]:
import pymupdf4llm

base_extraction_options = {
    "footer": False,
    "header": False,
    "show_progress": True,
}

md = pymupdf4llm.to_markdown(
    source,
    **base_extraction_options,
    # pages=[0],               # first page only
    # page_chunks=True,        # return per-page dictionaries
    # write_images=True,       # extract images to disk
    # image_path="assets/",    # image output directory
    # image_format="png",
)


Inspect the markdown (first 900 characters):


In [ ]:
print(f"Converted {len(md):,} characters")
print(md[:900])


You could also render it nicely:


In [ ]:
# Uncomment to display the markdown
# display(Markdown(md[:900]))

# Try it!
Turn that markdown into a **structured resume** with the same Outlines + Pydantic pattern you used on `long_review`.

1. Define nested models, for example:
   - `Role`: `title`, `organization`, `start`, `end`, `highlights: list[str]`
   - `Education`: `degree`, `institution`, `year`
   - `Resume`: `name`, `email`, `location`, `headline`, `skills: list[str]`, `experience: list[Role]`, `education: list[Education]`
2. Call `model` with a prompt that includes `md` and `Resume` as the output type (`max_new_tokens=400` is a reasonable start).
3. Validate with `Resume.model_validate_json` and print name, skills, and each role.

Adapt the fields to **your** CV. If a section is missing, keep the list empty rather than inventing facts.


In [ ]:
# INSERT YOUR CODE HERE


## Conclusion
- Outlines constrains generation so the output matches a type you pass in — no post-hoc parsing
- `Literal` is enough for closed-set labels such as sentiment
- Pydantic models capture nested records (ratings, lists, summaries) you can validate and use in code
- PDF4LLM turns a CV PDF into markdown the LLM can read; the same Pydantic pattern then extracts fields you can store or query
- Small instruction-tuned models work for this lab; larger models usually produce better field content
